démarrer l'API depuis le dépôt



In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q fastapi uvicorn
import subprocess, time, requests
import pandas as pd

DATA = '/content/drive/MyDrive/ProjetPCO/'
URL = 'http://127.0.0.1:8000'
log = open('/content/api.log', 'w')
subprocess.run(['pkill', '-f', 'uvicorn'])
server = subprocess.Popen(['uvicorn', 'api:app', '--port', '8000'],
                          cwd=DATA + 'leak-detection/api', stdout=log, stderr=log)
time.sleep(5)
print(requests.get(URL + '/counters').status_code)

les compteurs utilisables (ceux qui ont au moins une période NORMAL et une FUITE)

In [ ]:
counters = pd.DataFrame(requests.get(URL + '/counters').json())
counters = counters[counters['nb_days'] > 0]

usable = []
for cid in counters['counter_id']:
    labels = requests.get(f'{URL}/counters/{cid}/labels', params={'merge': True}).json()
    categories = {l['label_category'] for l in labels}
    if {'FUITE_CASSE', 'NORMAL_PERIOD'} <= categories:
        usable.append(cid)
print(len(usable), 'usable counters')

le jeu de données figé

In [ ]:
with_data = set(counters['counter_id'])
all_labels = pd.read_sql("SELECT DISTINCT counter_id FROM labeled_periods",
                         __import__('sqlite3').connect(DATA + 'leak_detection.db'))
with_labels = set(all_labels['counter_id'])
print('Data only   :', len(with_data - with_labels))
print('Labels only :', len(with_labels - with_data))
print('Both        :', len(with_data & with_labels))

In [ ]:
PRIORITY = ['NORMAL_PERIOD', 'BLANK', 'VOLUME_SERVICE', 'DEFAUT_MATERIEL', 'FUITE_CASSE']

def load_counter(cid):
    daily = pd.DataFrame(requests.get(f'{URL}/counters/{cid}/daily').json())
    daily['date'] = pd.to_datetime(daily['date'])
    daily['label'] = 'NONE'
    labels = requests.get(f'{URL}/counters/{cid}/labels', params={'merge': True}).json()
    labels.sort(key=lambda l: PRIORITY.index(l['label_category']) if l['label_category'] in PRIORITY else -1)
    for l in labels:
        in_period = daily['date'].between(l['start'], l['end'])
        daily.loc[in_period, 'label'] = l['label_category']
    daily['counter_id'] = cid
    return daily

dataset = pd.concat([load_counter(c) for c in usable], ignore_index=True)
!mkdir -p {DATA}model
dataset.to_csv(DATA + 'model/dataset_v1.csv', index=False)
print(dataset.shape)
print(dataset['label'].value_counts())

In [ ]:
summary = dataset.groupby('counter_id').agg(
    days=('date', 'size'),
    normal_days=('label', lambda s: (s == 'NORMAL_PERIOD').sum()),
    leak_days=('label', lambda s: (s == 'FUITE_CASSE').sum()),
)
print(summary.describe().round(0))
print(summary.sort_values('normal_days').head(10))

In [ ]:
for min_normal in [1, 14, 30, 60]:
    ok = summary[(summary['normal_days'] >= min_normal) & (summary['leak_days'] > 0)]
    print(f'>= {min_normal:2} normal days and >= 1 leak day: {len(ok)} counters')

Étape 6 : écrire l'évaluation. On écrit l'arbitre avant les joueurs.

Cellule A : garder les 38 compteurs et repérer les fuites

In [ ]:
keep = summary[(summary['normal_days'] >= 1) & (summary['leak_days'] > 0)].index
data = dataset[dataset['counter_id'].isin(keep)].sort_values(['counter_id', 'date']).copy()

# a leak event = a run of consecutive FUITE days on the same counter
is_leak = data['label'] == 'FUITE_CASSE'
new_event = is_leak & ~(is_leak.shift(fill_value=False) & (data['counter_id'] == data['counter_id'].shift()))
data['event_id'] = new_event.cumsum().where(is_leak)
print(len(keep), 'counters,', data['event_id'].nunique(), 'leak events')
# Time split per counter: oldest 70% of NORMAL days = train, the rest = eval
normal = data['label'] == 'NORMAL_PERIOD'
rank = data[normal].groupby('counter_id')['date'].rank(pct=True)
data['split'] = 'eval'
data.loc[rank[rank <= 0.7].index, 'split'] = 'train'
print(data['split'].value_counts())

Cellule B : la fonction d'évaluation

In [ ]:
# def evaluate(df, alert_col='alert'):
#     events = df[df['event_id'].notna()].groupby('event_id')[alert_col].any()

#     normal = df[(df['label'] == 'NORMAL_PERIOD') & (df['split'] == 'eval')]
#     return {
#         'detection_rate': round(events.mean(), 3),           # share of leaks caught
#         'false_alarm_rate': round(normal[alert_col].mean(), 3),  # share of normal days flagged
#     }

def evaluate(df, alert_col='alert', max_delay=3):
    leaks = df[df['event_id'].notna()].copy()
    leaks['day_in_event'] = leaks.groupby('event_id').cumcount()  # 0 = first leak day
    early = leaks[leaks['day_in_event'] < max_delay]
    events = early.groupby('event_id')[alert_col].any()
    normal = df[(df['label'] == 'NORMAL_PERIOD') & (df['split'] == 'eval')]
    return {'detection_rate': round(events.mean(), 3),
            'false_alarm_rate': round(normal[alert_col].mean(), 3)}

Cellule C : tester l'arbitre avec deux joueurs absurdes

In [ ]:
data['alert'] = True;  print('Always alert:', evaluate(data))
data['alert'] = False; print('Never alert :', evaluate(data))

In [ ]:
old = [100429, 100037, 100197, 100374, 100070, 100019, 100449]
print(summary.loc[summary.index.intersection(old)])

Étape 7 : le z-score. Ajoute cette cellule D :

In [ ]:
# Cell D: z-score baseline on daily_volume
train = data[data['split'] == 'train']
stats = train.groupby('counter_id')['daily_volume'].agg(['mean', 'std', 'count'])
print('Counters with stats:', stats['std'].notna().sum(), '/', len(keep))

mean = data['counter_id'].map(stats['mean'])
std = data['counter_id'].map(stats['std'])
data['z'] = (data['daily_volume'] - mean) / std

for k in [2, 3, 4]:
    data['alert'] = data['z'].abs() > k
    print(f'k={k}:', evaluate(data))

In [ ]:
# Cell E: random alerts at the same rate as z-score k=3
import numpy as np
rng = np.random.default_rng(0)
data['alert'] = rng.random(len(data)) < 0.086
print('Random 8.6%:', evaluate(data))

In [ ]:
# Cell F: results table, filled model after model
counters_26 = summary[(summary['normal_days'] >= 30) & (summary['leak_days'] > 0)].index
results = []

data['alert'] = data['z'].abs() > 3
results.append({'model': 'z-score k=3', 'counters': 38, **evaluate(data)})
sub = data[data['counter_id'].isin(counters_26)]
results.append({'model': 'z-score k=3', 'counters': 26, **evaluate(sub)})
pd.DataFrame(results)